In [1]:
import os
import torch
import random
import numpy as np
from PIL import Image
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

# For training and logging
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt


In [2]:
# ------------------
# Dataset Definition
# ------------------
class MyRatDataset(Dataset):
    """
    A dataset that reads images and YOLO-format annotations from a folder.
    Expected structure:
        root/
          images/
            img1.jpg
            img2.jpg
            ...
          labels/
            img1.txt
            img2.txt
            ...
    Each label file should contain lines (YOLO format):
        class_id  x_center_norm  y_center_norm  width_norm  height_norm
    Negative samples can be created on the fly with a given probability.
    This updated version prints a warning and ignores any image whose label file contains negative values.
    """
    def __init__(self, root, transforms=None, negative_sample_ratio=0.3):
        self.root = root
        self.transforms = transforms
        self.negative_sample_ratio = negative_sample_ratio
        
        self.img_dir = os.path.join(root, "images")
        self.lbl_dir = os.path.join(root, "labels")
        
        # Collect image file names
        self.imgs = [f for f in os.listdir(self.img_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.imgs.sort()

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.lbl_dir, label_name)
        
        boxes = []
        labels = []
        corrupt = False
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    try:
                        class_id = int(parts[0]) + 17  # shift if needed; reserve 0 for background
                        x_center_norm = float(parts[1])
                        y_center_norm = float(parts[2])
                        width_norm    = float(parts[3])
                        height_norm   = float(parts[4])
                    except ValueError:
                        continue

                    if x_center_norm < 0 or y_center_norm < 0 or width_norm < 0 or height_norm < 0:
                        # Print a warning and treat image as negative sample
                        print(f"WARNING {img_path}: ignoring corrupt image/label: negative label values "
                              f"[{x_center_norm:.5f}, {y_center_norm:.5f}, {width_norm:.5f}, {height_norm:.5f}]")
                        corrupt = True
                        break

                    x_center = x_center_norm * w
                    y_center = y_center_norm * h
                    box_width = width_norm * w
                    box_height = height_norm * h
                    x_min = x_center - box_width / 2
                    y_min = y_center - box_height / 2
                    x_max = x_center + box_width / 2
                    y_max = y_center + box_height / 2
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(class_id)
        
        if corrupt:
            boxes = []
            labels = []
        
        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Optionally generate a negative sample
        if boxes.shape[0] > 0 and random.random() < self.negative_sample_ratio:
            img = self.generate_negative_sample(img, boxes)
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        
        image_id = torch.tensor([idx])
        if boxes.size(0) > 0:
            area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        else:
            area = torch.empty((0,), dtype=torch.float32)
        iscrowd = torch.zeros((labels.shape[0],), dtype=torch.int64)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": image_id,
            "area": area,
            "iscrowd": iscrowd
        }
        
        if self.transforms:
            img = self.transforms(img)
        
        return img, target

    def generate_negative_sample(self, img, boxes):
        img_np = np.array(img)
        for box in boxes:
            x_min, y_min, x_max, y_max = box.int().tolist()
            x_min = max(0, x_min)
            y_min = max(0, y_min)
            x_max = min(img_np.shape[1], x_max)
            y_max = min(img_np.shape[0], y_max)
            if x_max > x_min and y_max > y_min:
                noise = np.random.randint(0, 256, (y_max - y_min, x_max - x_min, 3), dtype=np.uint8)
                img_np[y_min:y_max, x_min:x_max, :] = noise
        return Image.fromarray(img_np)


In [3]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models import VGG16_Weights

# Keep num_classes=91 so that the pretrained head (trained on COCO) remains valid.
num_classes = 91

model = ssd300_vgg16(
    weights=SSD300_VGG16_Weights.DEFAULT, 
    num_classes=num_classes, 
    weights_backbone=VGG16_Weights.IMAGENET1K_FEATURES
)


In [4]:
# Define transforms (resize to 300x300 to match SSD300 input)
transforms = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Create training and validation datasets
train_dataset = MyRatDataset(root="new_dataset/train", transforms=transforms, negative_sample_ratio=0.35)
val_dataset   = MyRatDataset(root="new_dataset/valid", transforms=transforms, negative_sample_ratio=0.35)

# Create dataloaders with an appropriate collate function
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=lambda batch: tuple(zip(*batch))
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=lambda batch: tuple(zip(*batch))
)

In [ ]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Set up optimizer and scheduler
optimizer = optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.005
)
num_epochs = 250
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
scaler = GradScaler()

# For logging
writer = SummaryWriter(log_dir="runs")

# Lists for plotting
train_losses = []
val_losses = []
train_cls_losses = []
val_cls_losses = []
train_bbox_losses = []
val_bbox_losses = []

# Helper: Freeze BatchNorm layers if desired.
def freeze_bn(module):
    if isinstance(module, torch.nn.BatchNorm2d):
        module.eval()

best_val_loss = float('inf')
best_epoch = -1

try:
    for epoch in range(num_epochs):
        model.train()
        model.apply(freeze_bn)
        
        epoch_train_loss = 0.0
        epoch_train_cls_loss = 0.0
        epoch_train_bbox_loss = 0.0
        
        for images, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False):
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # Optionally filter out samples with empty boxes (if needed)
            filtered_images = []
            filtered_targets = []
            for img, tgt in zip(images, targets):
                if tgt["boxes"].numel() > 0:
                    filtered_images.append(img)
                    filtered_targets.append(tgt)
            
            optimizer.zero_grad()
            with autocast():
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device, requires_grad=True)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["classification"]
                    bbox_loss = loss_dict["bbox_regression"]
                    total_loss = cls_loss + bbox_loss
            
            if total_loss.item() != 0:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            
            epoch_train_loss += total_loss.item()
            if len(filtered_images) > 0:
                epoch_train_cls_loss += cls_loss.item()
                epoch_train_bbox_loss += bbox_loss.item()
        
        epoch_train_loss /= len(train_loader)
        epoch_train_cls_loss /= len(train_loader)
        epoch_train_bbox_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        train_cls_losses.append(epoch_train_cls_loss)
        train_bbox_losses.append(epoch_train_bbox_loss)
        
        writer.add_scalar("Loss/Train/Total", epoch_train_loss, epoch)
        writer.add_scalar("Loss/Train/Class", epoch_train_cls_loss, epoch)
        writer.add_scalar("Loss/Train/BBox", epoch_train_bbox_loss, epoch)
        print(f"Epoch {epoch+1}/{num_epochs}: Train Total Loss: {epoch_train_loss:.4f}, "
              f"Train cls Loss: {epoch_train_cls_loss:.4f}, Train bbox Loss: {epoch_train_bbox_loss:.4f}")
        
        # Validation loop
        model.train()  # SSD returns losses only in train mode.
        model.apply(freeze_bn)
        
        epoch_val_loss = 0.0
        epoch_val_cls_loss = 0.0
        epoch_val_bbox_loss = 0.0
        
        with torch.no_grad():
            for images, targets in val_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
                
                filtered_images = []
                filtered_targets = []
                for img, tgt in zip(images, targets):
                    if tgt["boxes"].numel() > 0:
                        filtered_images.append(img)
                        filtered_targets.append(tgt)
                
                if len(filtered_images) == 0:
                    total_loss = torch.tensor(0., device=device)
                else:
                    loss_dict = model(filtered_images, filtered_targets)
                    cls_loss = loss_dict["classification"]
                    bbox_loss = loss_dict["bbox_regression"]
                    total_loss = cls_loss + bbox_loss
                
                epoch_val_loss += total_loss.item()
                if len(filtered_images) > 0:
                    epoch_val_cls_loss += cls_loss.item()
                    epoch_val_bbox_loss += bbox_loss.item()
        
        epoch_val_loss /= len(val_loader)
        epoch_val_cls_loss /= len(val_loader)
        epoch_val_bbox_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        val_cls_losses.append(epoch_val_cls_loss)
        val_bbox_losses.append(epoch_val_bbox_loss)
        
        writer.add_scalar("Loss/Val/Total", epoch_val_loss, epoch)
        writer.add_scalar("Loss/Val/Class", epoch_val_cls_loss, epoch)
        writer.add_scalar("Loss/Val/BBox", epoch_val_bbox_loss, epoch)
        print(f"Epoch {epoch+1}/{num_epochs}: Val Total Loss: {epoch_val_loss:.4f}, "
              f"Val cls Loss: {epoch_val_cls_loss:.4f}, Val bbox Loss: {epoch_val_bbox_loss:.4f}")
        
        # scheduler.step()
        
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), "Best_Model.pth")
            print(f"Best model saved at epoch {best_epoch} with val loss {best_val_loss:.4f}")
    
except KeyboardInterrupt:
    print("Training interrupted. Saving best model so far...")

torch.save(model.state_dict(), "Final_Model.pth")
print(f"Final model saved to Final_Model.pth")
print(f"Best model was from epoch {best_epoch} with val loss {best_val_loss:.4f}")


Epoch 211/250: Train Total Loss: 5.1372, Train cls Loss: 0.2727, Train bbox Loss: 4.8645
Epoch 211/250: Val Total Loss: 9.6334, Val cls Loss: 2.6489, Val bbox Loss: 6.9845


Epoch 212/250: Train Total Loss: 4.1815, Train cls Loss: 0.2241, Train bbox Loss: 3.9574
Epoch 212/250: Val Total Loss: 9.7122, Val cls Loss: 2.7442, Val bbox Loss: 6.9679


Epoch 213/250: Train Total Loss: 4.9212, Train cls Loss: 0.2819, Train bbox Loss: 4.6393
Epoch 213/250: Val Total Loss: 10.8689, Val cls Loss: 2.4754, Val bbox Loss: 8.3934


Epoch 214/250: Train Total Loss: 5.9075, Train cls Loss: 0.2374, Train bbox Loss: 5.6701
Epoch 214/250: Val Total Loss: 10.5841, Val cls Loss: 2.6656, Val bbox Loss: 7.9186


Epoch 215/250: Train Total Loss: 3.8567, Train cls Loss: 0.1953, Train bbox Loss: 3.6614
Epoch 215/250: Val Total Loss: 8.5931, Val cls Loss: 2.4747, Val bbox Loss: 6.1184


...

Epoch 247/250: Train Total Loss: 5.6619, Train cls Loss: 0.3877, Train bbox Loss: 5.2742
Epoch 247/250: Val Total Loss: 10.7663, Val cls Loss: 3.0029, Val bbox Loss: 7.7634


Epoch 248/250: Train Total Loss: 5.5158, Train cls Loss: 0.3062, Train bbox Loss: 5.2096
Epoch 248/250: Val Total Loss: 10.0731, Val cls Loss: 2.3632, Val bbox Loss: 7.7099


Epoch 249/250: Train Total Loss: 6.3097, Train cls Loss: 0.2989, Train bbox Loss: 6.0108
Epoch 249/250: Val Total Loss: 9.4629, Val cls Loss: 2.6531, Val bbox Loss: 6.8098


Epoch 250/250: Train Total Loss: 4.9688, Train cls Loss: 0.2289, Train bbox Loss: 4.7399
Epoch 250/250: Val Total Loss: 11.8127, Val cls Loss: 2.4282, Val bbox Loss: 9.3845
Final model saved to Final_Model.pth
Best model was from epoch 131 with val loss 7.6261


: 

In [ ]:
# ---------------------------
# Plot Training and Validation Losses
# ---------------------------
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_losses, label="Train Total Loss")
plt.plot(range(1, num_epochs+1), val_losses, label="Val Total Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation Total Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_cls_losses, label="Train Classification Loss")
plt.plot(range(1, num_epochs+1), val_cls_losses, label="Val Classification Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation Classification Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs+1), train_bbox_losses, label="Train BBox Loss")
plt.plot(range(1, num_epochs+1), val_bbox_losses, label="Val BBox Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs. Validation BBox Loss")
plt.legend()
plt.grid(True)
plt.show()

In [12]:
import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image

# from torchvision.models import VGG16_Weightsts
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

# Define the transform (same as during training)
transforms = T.Compose([
    T.Resize((320, 320)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda")


# Keep num_classes=91 so that the pretrained head (trained on COCO) remains valid.
num_classes = 91

model = ssd300_vgg16(
    weights=SSD300_VGG16_Weights.DEFAULT, 
    num_classes=num_classes, 
    # weights_backbone=VGG16_Weights.IMAGENET1K_FEATURES
)

model.to(device)

# Load saved weights
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Best_Model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

def predict(image_input, model, device, threshold=0.7, nms_threshold=0.3, top_k=1):
    """
    Predict detections on a given image.
    
    Args:
        image_input (str or np.ndarray): If string, treated as file path;
                                           if np.ndarray, treated as an OpenCV BGR image.
        model: The detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): IoU threshold for non-maximum suppression.
        top_k (int): Number of top detections to display.
    
    Returns:
        orig_img (np.ndarray): The original image in BGR format.
        boxes (np.ndarray): Array of bounding boxes.
        scores (np.ndarray): Detection scores.
        labels (np.ndarray): Detected labels.
        inference_time_ms (float): Inference time in milliseconds.
    """
    # Check input type and convert accordingly
    if isinstance(image_input, np.ndarray):
        # image_input is a frame (BGR)
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()  # keep a copy in BGR for display
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported type for image_input. Must be str or np.ndarray.")
    
    # Apply transforms to create a tensor
    img_tensor = transforms(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)  # add batch dimension

    # Measure inference time using OpenCV ticks
    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time_ms = (end - start) / cv2.getTickFrequency() * 1000.0

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()
    scores = output['scores'].cpu().numpy()
    labels = output['labels'].cpu().numpy()

    # Filter out detections below confidence threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    if len(boxes) > 0:
        # Optionally, apply non-maximum suppression (NMS)
        if nms_threshold > 0:
            boxes_tensor = torch.tensor(boxes, device=device)
            scores_tensor = torch.tensor(scores, device=device)
            labels_tensor = torch.tensor(labels, device=device)
            keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, nms_threshold)
            keep_indices = keep_indices.cpu().numpy()
            boxes = boxes_tensor[keep_indices].cpu().numpy()
            scores = scores_tensor[keep_indices].cpu().numpy()
            labels = labels_tensor[keep_indices].cpu().numpy()

        # Select top_k detections based on scores
        k = min(top_k, len(scores))
        sorted_indices = np.argsort(scores)[::-1][:k]
        boxes = boxes[sorted_indices]
        scores = scores[sorted_indices]
        labels = labels[sorted_indices]

    return orig_img, boxes, scores, labels, inference_time_ms

def main():
    # Choose one of your video files
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\3_Mice.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_video.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Cohort_1.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\random_youtube_video.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop.avi"
    video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop2.avi"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\desktop3.avi"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\brown_rats.mp4"
    # video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\BaselineDark.mp4"
    top_k = 3
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file: {video_path}")
        return

    frame_count = 0  # Initialize a frame counter

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1  # Increment the frame counter
        
        # Predict on the current frame
        orig_img, boxes, scores, labels, inf_time = predict(
            frame, model, device, threshold=0.3, nms_threshold=0.1, top_k=top_k)
        print("Labels:", labels)
        print("Inference time (ms):", inf_time)

        # Draw bounding boxes, label text, and centroid red dot on the frame
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(orig_img, f"Rat: {score:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            # Compute and draw the centroid as a red dot
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            cv2.circle(orig_img, (cx, cy), 3, (0, 0, 255), -1)
        
        # Draw inference time on the frame
        cv2.putText(orig_img, f"Inference: {inf_time:.1f} ms", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        
        # Display the frame number (e.g., at the top left corner)
        cv2.putText(orig_img, f"Frame: {frame_count}", (10, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)
        
        # Display the frame with predictions
        cv2.imshow("Predictions", orig_img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()


Labels: []
Inference time (ms): 165.60420000000002
Labels: []
Inference time (ms): 63.0019
Labels: []
Inference time (ms): 54.7861
Labels: []
Inference time (ms): 55.760400000000004
Labels: []
Inference time (ms): 58.9169
Labels: []
Inference time (ms): 58.573600000000006
Labels: []
Inference time (ms): 51.4234
Labels: []
Inference time (ms): 52.8116
Labels: [17 17]
Inference time (ms): 49.9737
Labels: [17 17]
Inference time (ms): 49.7263
Labels: []
Inference time (ms): 99.5287
Labels: []
Inference time (ms): 50.0074
Labels: []
Inference time (ms): 49.705600000000004
Labels: []
Inference time (ms): 50.9697
Labels: []
Inference time (ms): 52.545500000000004
Labels: []
Inference time (ms): 49.7287
Labels: []
Inference time (ms): 50.6471
Labels: []
Inference time (ms): 52.1259
Labels: []
Inference time (ms): 52.303799999999995
Labels: [17]
Inference time (ms): 49.9485
Labels: [17]
Inference time (ms): 50.4769
Labels: []
Inference time (ms): 50.6409
Labels: []
Inference time (ms): 51.2431
